In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv
import urllib.parse
import os
import clickhouse_connect
import numpy as np
import pandas as pd
import datetime as dt

orange = '#FF670F'
pink = '#FC006D'
purple = '#8659D9'
blue = '#006CED'
green = '#2BD67C'
red = '#FF5D5D'
black = '#2B3034'

#tables
releves='datawarehouse_prod.fact_payload'
enedis = 'datamart_nrj.payload_enedis'
clients = 'datawarehouse_prod.dim_entity'
equipements = 'datawarehouse_prod.dim_device'
capteurs = 'datawarehouse_prod.dim_sensor'
patrimoine = 'datawarehouse_prod.device_patrimonial_data'

#autres variables
source = 'enedis'
maxi = 10000000000000

load_dotenv()
host = urllib.parse.quote_plus(os.environ.get("clk_prod"))
username = urllib.parse.quote_plus(os.environ.get("USER_NAME"))
password = urllib.parse.quote_plus(os.environ.get('clk_prod_pass'))
client = clickhouse_connect.get_client(host=host, username=username,
                                       port = 443, password=password)

<span style="color:red">

## Points à vérifier
1. Creuser tout ce qui tourne autour des natures / sous-natures / typologies
3. Quel pas de temps sur le capteur "Cons" (_Courbe de charge de puissance puissance active instantanée (W)_) par segment de PDL ?
        - Complétude par client
</span>

# <span style="color:purple"> 1 - Volume, valeurs, fréquences des données </span>

## <span style="color:green"> 1- Distribution  des relevés par client </span> 

In [ ]:
df = client.query_df(f"""SELECT p.entity_id, p.device_id, p.payload_metric_id, count(p.payload_timestamp) nbPay\
                            FROM {releves} p
                            WHERE p.payload_source = '{source}'
                            GROUP BY p.entity_id, p.device_id, p.payload_metric_id\
                            ORDER BY nbPay
                            settings max_rows_to_read = {maxi}""")

In [ ]:
cnt = len(df.entity_id.unique())
avg = df.groupby('entity_id').nbPay.sum().mean().astype(int)
avg2 = df.groupby('entity_id').nbPay.count().mean().astype(int)
avg3 = df.groupby(['entity_id', 'device_id'], group_keys=False).count().groupby('entity_id').nbPay.count().mean().astype(int)
avg4 = df.groupby(['entity_id', 'payload_metric_id'], group_keys=False).count().groupby('entity_id').nbPay.count().mean().astype(int)
print(f"Nombre total de clients: {cnt}.")
print(f"Moyenne du nombre de relevés par client: {avg}.")
print(f"Moyenne du nombre de couples 'equipement-capteur' par client: {avg2}.")
print(f"Moyenne du nombre d'équipements par client: {avg3}.")
print(f"Moyenne du nombre de capteurs par client {avg4}.")

In [ ]:
avg = df.groupby('entity_id').nbPay.sum()
avg2 = df.groupby('entity_id').nbPay.count()
avg3 = df.groupby(['entity_id', 'device_id'], group_keys=False).count().groupby('entity_id').nbPay.count()
avg4 = df.groupby(['entity_id', 'payload_metric_id'], group_keys=False).count().groupby('entity_id').nbPay.count()
fig = go.Figure()
fig.add_trace(go.Box(name="Nombre de relevés moyen.", y=avg, marker={'color': blue}))
fig.add_trace(go.Box(name="Nombre de couple 'equip-capt'.", y=avg2, marker={'color': blue}))
fig.add_trace(go.Box(name="Nombre d'équipements moyen.", y=avg3, marker={'color': blue}))
fig.add_trace(go.Box(name = "Nombre de capteurs moyen.", y=avg4, marker={'color': blue}))
fig.update_yaxes(type="log")
fig.update_layout(title=f"Distribution des relevés / équipements / capteurs par client.",
              xaxis_title="Type de mesure par client", 
              yaxis_title="Nombre moyen <br> (Echelle logarithmique)",
              font_family="Arial", font_color=black, title_font_family="Times New Roman", 
                title_font_color=black, width=1000, height=400,
                template="simple_white", showlegend=False)
fig.show()

## <span style="color:green"> 2- Distribution par famille d'équipement </span> 

In [ ]:
df_familles = client.query_df(f"""
SELECT d.device_type, count(p.payload_timestamp) nbPay
FROM {releves} p
INNER JOIN (SELECT device_type, device_id
		FROM {equipements}) d	
ON p.device_id = d.device_id
WHERE p.payload_source = '{source}'
GROUP BY d.device_type
settings max_rows_to_read = {maxi};""")

In [ ]:
df_familles.

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(x = df_familles.device_type, y=df_familles.nbPay, text = df_familles.nbPay,
        marker={'color': blue}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Distribution des relevés enedis en fonction de la famille d'équipement",
                  xaxis_title="Famille d'équipement", 
                  yaxis_title="Nombre de relevés",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

## <span style="color:green"> 3- Distribution par capteur </span> 

In [ ]:
df_sens = client.query_df(f"""SELECT 
                            d.sensor_id, d.name, p.payload_metric_id,
                            countDistinct(p.device_id) nbDev, 
                            count(p.payload_timestamp) nbPay
                        FROM {releves} p
                        LEFT JOIN {capteurs} AS d
                            ON p.payload_metric_id = d.sensor_id
                        WHERE p.payload_source = '{source}'
                        GROUP BY d.sensor_id, p.payload_metric_id, d.name
                        ORDER BY nbPay
                        settings max_rows_to_read = {maxi}""")

In [ ]:
df_sens[df_sens.sensor_id==""].payload_metric_id.to_list()

In [ ]:
print(f"Moyenne du nombre d'équipements par capteur: {int(df_sens.nbDev.mean())}.")
print(f"Moyenne du nombre de relevés par capteur {int(df_sens.nbPay.mean())}.")

In [ ]:
df = client.query_df(f"""SELECT
                            d.unit,
                            countDistinct(p.payload_metric_id) sens,
                            countDistinct(p.device_id)         dev
                        FROM {releves} p
                        LEFT JOIN {capteurs} d
                            ON p.payload_metric_id = d.sensor_id
                        WHERE p.payload_source = 'enedis'
                        GROUP BY d.unit
                        settings max_rows_to_read=10000000000000;""")

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(name='# equipements', x=df.unit, y=df.dev, text=df.dev,
        marker={'color': blue}
    ))
fig.add_trace(
    go.Bar(name='# capteurs', x=df.unit, y=df.sens, text=df.sens,
        marker={'color': pink}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Distribution des capteurs et équipements enedis en fonction de l'unité des capteurs",
                  xaxis_title="Unité du capteur", 
                  yaxis_title="Nombre de capteurs / équipements <br> (Echelle logarithmique)",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
print(f"""
    Certains capteurs ({int(df[(df.unit=="")|(df.unit=='-')].shape[0]/df.shape[0]*100)}%) n'ont pas d'unité.
    Seulement 3 capteurs rapportent des relevés en kWh et 33 rapportent de l'énergie réactive""")

## <span style="color:green"> 4- Distribution par combinaison équipement - capteur </span> 

In [ ]:
df = client.query_df(f"""SELECT p.device_id, p.payload_metric_id, count(p.payload_timestamp) nbPay\
                            FROM {releves} p\
                            WHERE p.payload_source = 'enedis'\
                            GROUP BY p.device_id, p.payload_metric_id\
                            ORDER BY nbPay
                            settings max_rows_to_read = {maxi}""")

In [ ]:
print(f"Nombre total de couples 'equipement-capteur': {df.shape[0]}. En moyenne {df.nbPay.mean().astype(int)} relevés par couple.")
cnt = len(df.device_id.unique())
avg = df.groupby('device_id').nbPay.sum().mean().astype(int)
avg2 = df.groupby('device_id').nbPay.count().mean().astype(int)
print(f"Nombre total d'équipements {cnt}, qui en moyenne ont {avg} relevés et sont reliés à {avg2} capteurs.")
cnt = len(df.payload_metric_id.unique())
avg = df.groupby('payload_metric_id').nbPay.sum().mean().astype(int)
avg2 = df.groupby('payload_metric_id').nbPay.count().mean().astype(int)
print(f"Nombre de capteurs: {cnt}, qui en moyenne ont {avg} relevés et sont reliés {avg2} equipements.")

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Histogram(nbinsx = 200, x = df.nbPay.sort_values(),
        marker={'color': blue}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Distribution des relevés en fonction des paires 'équipement - capteur'",
                  xaxis_title="Nombre de relevés", 
                  yaxis_title="Nombre de paires 'équipement - capteur' <br> (Echelle logarithmique)",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

## <span style="color:green"> 5- Distribution par typologie de bâtiment </span> 

In [ ]:
df_build = client.query_df(f"""
        SELECT b.site_typology,
                b.sub_nature, 
                count(distinct p.entity_id) ent, 
                count(distinct p.device_id) dev, 
                count(distinct p.payload_metric_id) sens, 
                count(distinct p.payload_timestamp) nbPay
        FROM {releves} p
        INNER JOIN (SELECT sub_nature, site_typology, device_id
        		FROM {patrimoine}) b
        ON b.device_id = p.device_id
        WHERE p.payload_source = '{source}'
        GROUP BY b.site_typology, b.sub_nature
        settings max_rows_to_read = {maxi};""")
df_build.sort_values('ent')

In [ ]:
df_typo = client.query_df(f"""
        SELECT b.site_typology,
                b.site_nature,
                b.sub_nature,
                count(distinct b.device_id) dev
        FROM {patrimoine} b
        GROUP BY b.site_typology, b.site_nature, b.sub_nature
        settings max_rows_to_read = {maxi};""")

In [ ]:
df_typo.sort_values('dev')

In [ ]:
df_typo.groupby('site_nature').count().sort_values('dev').tail(20)

<span style="color:orange"> Il y existe un certain nombre d'équipements dont la typologie du site ou la sous nature n'est pas renseignée. Dans le pire des cas, il existe **291 clients** et **26647 équipements** qui n'ont aucun de ces deux champs renseignés </span>

In [ ]:
df_build.groupby('site_typology')[['ent', 'dev', 'sens', 'nbPay']].sum().sort_values('ent', ascending=False)

# <span style="color:purple"> 2 - Exploration par segment </span>

## <span style="color:green"> 1 - Volumes de clients et relevés par segment </span>

In [ ]:
df_seg = client.query_df(f"""SELECT device_id, segment, count(payload_value) nbPay, max(payload_timestamp_tz) timestamp
FROM {enedis}
GROUP BY device_id, segment
settings max_rows_to_read = 10000000000000""")

Certains PDL ont changé de segment au cours du temps

In [ ]:
nb_segments = df_seg.groupby('device_id').agg({'segment':'count', 'timestamp':lambda x: [str(e)[:10] for e in x.sort_values()]}).sort_values('segment')
#les 10 PDL ayant le plus segements
nb_segments.tail(10)

In [ ]:
count = df_seg.groupby('segment').count()
fig = go.Figure()
fig.add_trace(
    go.Bar(x=count.index, y=count.nbPay,
        marker={'color': pink}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Nombre de relevés par segment",
                  xaxis_title="Segment PDL", 
                  yaxis_title="Nombre de relevés (échelle logarithmique)",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
df_seg = client.query_df(f"""SELECT e.segment, t.entity_name, count(payload_value) nbPay
FROM {enedis} e
JOIN (
    SELECT 
        d.device_id, 
        c.entity_name
    FROM {equipements} AS d
    JOIN {clients} AS c
        ON d.iot_entity = c.entity_id
    ) t
    ON e.device_id = t.device_id
GROUP BY e.segment, t.entity_name
settings max_rows_to_read = {maxi}""")

In [ ]:
counts = df_seg.groupby('segment').entity_name.count()
fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': blue}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Nombre de clients par segment",
                  xaxis_title="Segment PDL", 
                  yaxis_title="Nombre de clients (échelle logarithmique)",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

## <span style="color:green"> 2 - Couverture des données sur les segment connus </span>

In [ ]:
df_couv = client.query_df(f"""SELECT t.entity_name, e.segment, min(e.payload_timestamp_tz) debut, max(e.payload_timestamp_tz) fin
FROM {enedis} e
JOIN (
    SELECT 
        d.device_id, 
        c.entity_name
    FROM {equipements} AS d
    JOIN {clients} AS c
        ON d.iot_entity = c.entity_id
    ) t
    ON e.device_id = t.device_id
WHERE NOT match(e.segment, 'Non Transmis')
GROUP BY t.entity_name, e.segment
settings max_rows_to_read = {maxi}""")

In [ ]:
df_couv

In [ ]:
date_range = pd.date_range(start=df_couv.debut.min(), end=df_couv.fin.max(), freq="MS")
time_coverage_df = []
for seg, s in df_couv[df_couv.segment.isin(['C1', 'C2', 'C3', 'C4', 'C5'])].groupby("segment"):
    for ma_date in date_range:    
        nbDev = s[(s.debut < ma_date) &  (s.fin > ma_date)].shape[0]
        time_coverage_df.append({
            'date':ma_date, "nbCouples":nbDev, "segment":seg})
time_coverage_df = pd.DataFrame(time_coverage_df)
fig = go.Figure()
for seg, s in time_coverage_df.groupby("segment"):
    fig.add_trace(go.Histogram(name=seg, histfunc='sum', 
                                 x=s.date, y=s.nbCouples))
# Set xaxis range
fig.update_xaxes(type='date', 
                 tickformat='%Y-%m-%d', 
                 nticks=10, 
                 range=[df_couv.debut.min() - pd.Timedelta(days=60), df_couv.fin.max() + pd.Timedelta(days=60)])
fig.update_yaxes(type='log')
fig.update_layout(title=f"Couverture des clients par segment de consommation ",
                  xaxis_title="Historique", 
                  yaxis_title="# clients",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
time_coverage_df = []
for seg, s in df_couv[~df_couv.segment.isin(['C1', 'C2', 'C3', 'C4', 'C5'])].groupby("segment"):
    for ma_date in date_range:    
        nbDev = s[(s.debut < ma_date) &  (s.fin > ma_date)].shape[0]
        time_coverage_df.append({
            'date':ma_date, "nbCouples":nbDev, "segment":seg})
time_coverage_df = pd.DataFrame(time_coverage_df)
fig = go.Figure()
for seg, s in time_coverage_df.groupby("segment"):
    fig.add_trace(go.Histogram(name=seg, histfunc='sum', 
                                 x=s.date, y=s.nbCouples))
# Set xaxis range
fig.update_xaxes(type='date', 
                 tickformat='%Y-%m-%d', 
                 nticks=10, 
                 range=[df_couv.debut.min() - pd.Timedelta(days=60), df_couv.fin.max() + pd.Timedelta(days=60)])
fig.update_yaxes(type='log')
fig.update_layout(title=f"Couverture des clients par segment de consommation ",
                  xaxis_title="Historique", 
                  yaxis_title="# clients",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

# <span style="color:purple"> 3 - Exploration des données de la famille 'Compteur Électricité' </span>

In [ ]:
famille = 'Compteur Électricité'

## <span style="color:green"> 1 - Distribution des valeurs des relevés </span> 

In [ ]:
# df = client.query_df(f"""
#     SELECT d.device_name, d.device_id, count(p.payload_timestamp) nbPay
#     FROM {releves} p
#     INNER JOIN (SELECT device_type, device_name, device_id
#     		FROM {equipements}
#             WHERE device_type = '{famille}') d
#     ON p.device_id = d.device_id
#     WHERE p.payload_source = 'enedis'
#     GROUP BY d.device_name, d.device_id
#     settings max_rows_to_read = 10000000000000;""")
# df.to_csv('./data/enedis_compteur_electricite_nbPay.csv')
df = pd.read_csv('./data/enedis_compteur_electricite_nbPay.csv')

### <span style="color:blue"> a)-Relevés issus des 10 couples équipements-capteur les plus actifs</span>

In [ ]:
smp = df.sort_values('nbPay', ascending=False).head(10)['d.device_id'].to_list()
df = client.query_df(f"""SELECT 
                            d.device_type, d.device_id, d.device_name, 
                            p.payload_metric_id, p.payload_timestamp, 
                            p.ingested_at, p.payload_value
                        FROM {releves} p
                        INNER JOIN (
                            SELECT device_type, device_id, device_name
                        	FROM {equipements}) d
                        ON p.device_id = d.device_id
                        WHERE p.device_id IN {smp} 
                            AND p.payload_timestamp > toDateTime('2024-01-01 12:00:00')
                        settings max_rows_to_read = {maxi}""")
df.to_csv('./data/enedis_payloads_sample.csv', index=False)
# df = pd.read_csv('./data/enedis_payloads_sample.csv')
df['payload_value'] = df.payload_value.apply(int)
df

In [ ]:
## Nb relevés
df_grp = df.groupby(['device_name', 'payload_metric_id']).payload_timestamp.count().reset_index()
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.02)
for i in df_grp.device_name.unique():
    s = df_grp[df_grp['device_name']==i]
    fig.add_trace(go.Violin(x=s.device_name,
                        y=s.payload_timestamp,
                        name=i,
                        box_visible=False,
                        meanline_visible=True), row=1, col=1)

## Valeurs des relevés
df_grp = df.groupby(['device_name', 'payload_metric_id']).payload_value.mean().reset_index()
for i in df_grp.device_name.unique():
    s = df_grp[df_grp['device_name']==i]
    fig.add_trace(go.Violin(x=s.device_name,
                        y=s.payload_value,
                        showlegend=False,
                        name=i,
                        box_visible=False,
                        meanline_visible=True), row=2, col=1)
## Fréquence des relevés
df_lag = []
for p_id, s in df.groupby(['device_name', 'payload_metric_id']):
    s = s.sort_values('payload_timestamp')
    df_lag.append({"device_name":p_id[0], "payload_metric_id":p_id[1], 
                  "lag": (s.payload_timestamp - s.payload_timestamp.shift()).mean()})
df_lag = pd.DataFrame(df_lag)
for i in df_lag.device_name.unique():
    s = df_lag[df_lag['device_name']==i]
    fig.add_trace(go.Violin(x=s.device_name,
                        y=s.lag + pd.to_datetime('1970/01/01'),
                        showlegend=False,
                        name=i,
                        box_visible=True,
                        meanline_visible=True), row=3, col=1)
fig.update_yaxes(title_text="Effectif moyen des relevés <br> par capteur.", row=1, col=1, type="log")
fig.update_yaxes(title_text="Valeur moyenne des relevés <br> par capteur.", row=2, col=1, type="log")
fig.update_yaxes(title_text="Fréquence moyenne des relevés <br> par capteur.", row=3, col=1, tickformat="%H:%M:%S.%f")
fig.update_xaxes(title_text="Nom de l'équipement", row=3, col=1)
fig.update_layout(title=f"Distribution des moyennes de fréquence des relevés par capteur en fonction des équipements.",
                    font_family="Arial", font_color=black,
                    title_font_family="Times New Roman",
                    title_font_color=black,
                    width=1000, height=1000,
                    template="simple_white")
fig.show()

### <span style="color:blue"> b)-Relevés issus de capteurs dont l'unité est en kWh</span>

In [ ]:
df_kwh = client.query_df(f"""SELECT
                            p.device_id,
                            p.payload_metric_id,
                            p.payload_timestamp,
                            p.payload_value,
                        FROM {releves} AS p
                        LEFT JOIN {capteurs} AS d
                            ON p.payload_metric_id = d.sensor_id
                        LEFT JOIN {equipements} AS e
                            ON p.device_id = e.device_id
                        WHERE p.payload_source = 'enedis'
                            AND match(d.unit, '(?i)^kwh$')
                            AND e.device_type = '{famille}'
                        settings max_rows_to_read=10000000000000;""")
df_kwh['payload_value'] = df_kwh.payload_value.apply(int)

In [ ]:
def box_stats(x):
    q1 = x.quantile(0.25)
    q2 = x.median()
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return pd.Series({
        "q1": q1,
        "median": q2,
        "q3": q3,
        "lowerfence": lower,
        "upperfence": upper,
        "mean": x.mean()
    })

box_kwh = df_kwh.groupby(["p.device_id","payload_metric_id"])["payload_value"].apply(box_stats).reset_index()

box_kwh = box_kwh.pivot_table(columns='level_2', values='payload_value', index=['p.device_id', 'payload_metric_id'])

fig = go.Figure()
fig.add_trace(go.Box(q1=box_kwh.q1, median=box_kwh['median'],
                  q3=box_kwh.q3, lowerfence=box_kwh.lowerfence,
                  upperfence=box_kwh.upperfence, mean=box_kwh['mean']))
fig.update_yaxes(type="log", range=(1,10))
fig.update_layout(title=f"Répartition des valeurs des relevés par couples équipement-capteur",
                  xaxis_title="Couple équipement capteur",\
                    yaxis_title="Valeur du relevé (kwh)", \
                    font_family="Arial", font_color=black, \
                    title_font_family="Times New Roman", \
                    title_font_color=black, width=1000, \
                    height=400, template="simple_white")
fig.show()

## <span style="color:green"> 2 - Description des fréquences des relevés </span>

### <span style="color:blue"> a) - Distribution des couples [capteur-equipement] en fonction de la fréquence de leurs relevés</span>

In [ ]:
df_freq = client.query_df(f"""
    SELECT p.device_id, p.payload_metric_id, arrayReduce('avg', diffs) AS avgDiff 
    FROM( 
        SELECT 
            p.device_id, 
            p.payload_metric_id, 
            arrayDifference(arraySort(groupArray(p.payload_timestamp))) AS diffs
        FROM {releves} p
        INNER JOIN {equipements} d 
        ON p.device_id = d.device_id
        WHERE d.device_type = '{famille}' AND p.payload_source = '{source}'
    GROUP BY 
        p.device_id, 
        p.payload_metric_id) p
    ORDER BY avgDiff DESC
    settings max_rows_to_read = {maxi}""")

In [ ]:
df_freq['timedelta'] = df_freq.avgDiff.apply(lambda x: dt.timedelta(seconds=x))
max_val = df_freq.head(100).avgDiff.max()
tickvals = [-float('inf'), 1, 60, 
            60*5, 60*10, 60*30, 60*45, 
            3600, 3600*6, 3600*12, 
            86400, 86400*5, 86400*15, 86400*30, max_val]
ticktext = ['< 1s', '< 1mn', '(1mn - 5mn])', '(5mn - 10mn])', '(10mn - 30mn])', 
           '(30mn - 45mn])', '(45mn - 1h])', '(1h - 6h])', '(6h - 12h])', 
           '(12h - 1jour])', '(1jour - 5jours])', '(5jours - 15jours])', 
           '(15jours - 30jours])', '> 1mois']
df_freq["interval"] = pd.cut(df_freq["avgDiff"], bins=tickvals, labels=ticktext)
counts = df_freq["interval"].value_counts(sort=False)

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': pink}
    )
)
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Fréquence moyenne des relevés par couples équipement-capteur",
                  xaxis_title="fréquence moyenne", 
                  yaxis_title="Nombre de couples <br> equipement-capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

<span style="color:orange">  En majorité, les équipements reçoivent des relevés à la journée </span>

### <span style="color:blue"> b) - Couples [capteur-equipement] dont la fréquence des relevés est <1s</span>

In [ ]:
no_freq_data = []
for dev, s in df_freq[df_freq.avgDiff <1].groupby('device_id'):
    sens_l = s.payload_metric_id.to_list()
    q = f"""
    SELECT device_id, payload_metric_id, count(*)
    FROM datawarehouse_prod.fact_payload 
    WHERE device_id='{dev}' AND payload_metric_id IN {sens_l}
    GROUP BY device_id, payload_metric_id
    settings max_rows_to_read = {maxi};"""
    no_freq_data.append(client.query_df(q))
no_freq_data = pd.concat(no_freq_data)
no_freq_data

<span style="color:orange">  21355 couples [capteur-equipement] sur 898448 (2%) n'ont qu'un seul relevé dans la table des payloads  </span>

### <span style="color:blue"> c) - Distribution des fréquences parmi les capteurs dont l'unité est du kwh</span>

In [ ]:
freq_kwh = client.query_df(f"""
SELECT p.device_id, p.payload_metric_id, arrayReduce('avg', diffs) AS avgDiff 
    FROM( 
        SELECT 
            p.device_id, 
            p.payload_metric_id, 
            arrayDifference(arraySort(groupArray(p.payload_timestamp))) AS diffs
        FROM {releves} p
        INNER JOIN {equipements} d 
        ON p.device_id = d.device_id
        INNER JOIN {capteurs} c
        ON p.payload_metric_id = c.sensor_id
        WHERE d.device_type = '{famille}' 
            AND p.payload_source = '{source}'
            AND match (c.unit, '(?i)^kwh$')
    GROUP BY 
        p.device_id, 
        p.payload_metric_id) p
    ORDER BY avgDiff DESC
    settings max_rows_to_read = {maxi}""")

In [ ]:
freq_kwh['timedelta'] = freq_kwh.avgDiff.apply(lambda x: dt.timedelta(seconds=x))
max_val = freq_kwh.head(100).avgDiff.max()
tickvals = [-float('inf'), 1, 60, 
            60*5, 60*10, 60*30, max_val]
ticktext = ['< 1s', '< 1mn', '(1mn - 5mn])', '(5mn - 10mn])', '(10mn - 30mn])', '<1h']
freq_kwh["interval"] = pd.cut(freq_kwh["avgDiff"], bins=tickvals, labels=ticktext)
counts = freq_kwh["interval"].value_counts(sort=False)

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': pink}
    )
)
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Fréquence moyenne des relevés par couples équipement-capteur",
                  xaxis_title="fréquence moyenne", 
                  yaxis_title="Nombre de couples <br> equipement-capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

## <span style="color:green"> 3 - Profondeur de l'historique des relevés </span>

In [ ]:
histo = client.query_df(f"""
        SELECT device_id, payload_metric_id, debut, fin, fin-debut AS couverture
        FROM(
            SELECT 
                p.device_id, 
                p.payload_metric_id, 
                min(p.payload_timestamp) debut,
                max(p.payload_timestamp) fin
            FROM {releves} p
            INNER JOIN {equipements} d
            ON p.device_id = d.device_id
            WHERE d.device_type = '{famille}' 
                AND p.payload_source = '{source}'
        GROUP BY 
            p.device_id, 
            p.payload_metric_id)
        settings max_rows_to_read = {maxi};""")

In [ ]:
date_range = pd.date_range(start=histo.debut.min(), end=histo.fin.max(), freq="MS")
time_coverage_df = []
for ma_date in date_range:
    nbDev = histo[(histo.debut < ma_date) &  (histo.fin > ma_date)].shape[0]
    time_coverage_df.append({
        'date':ma_date, "nbCouples":nbDev})
time_coverage_df = pd.DataFrame(time_coverage_df)
fig = go.Figure(go.Histogram(histfunc='sum', 
                             x=time_coverage_df.date, y=time_coverage_df.nbCouples, 
                            marker={'color': green}))
# Set xaxis range
fig.update_xaxes(type='date', 
                 tickformat='%Y-%m-%d', 
                 nticks=10, 
                 range=[histo.debut.min(), histo.fin.max()])
fig.update_yaxes(type='log')
fig.update_layout(title=f"Distribution des couples 'equipement-capteur' sur le temps.",
                  xaxis_title="Date", 
                  yaxis_title="# couples 'equipement-capteur' <br> (Echelle logarithmique)",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
histo['timedelta'] = histo.couverture.apply(lambda x: dt.timedelta(seconds=x))
max_val = histo.head(100).couverture.max()
tickvals = [histo.couverture.min(), 60*60*24,
            86400*30, 2592000*6, 2592000*12, 
            31104000*2, 31104000*3, 31104000*4, max_val]
ticktext = ['< 1 journée', '(1j - 1 mois])', '(1 mois - 6 mois])', '(6 mois - 1 an])', 
            '(1 an - 2 ans])', '(2 ans - 3 ans])', '(3 ans - 4 ans])', '> 4 ans']
histo["interval"] = pd.cut(histo["couverture"], bins=tickvals, labels=ticktext)
counts = histo["interval"].value_counts(sort=False)

fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': pink}
    )
)
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Couverture 'théorique' des relevés par couples équipement-capteur",
                  xaxis_title="Couverture théorique", 
                  yaxis_title="Nombre de couples <br> equipement-capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

## <span style="color:green"> 4 - Couverture des relevés - en kwh uniquement (étude des plages continues) <br>Relevés parus après 01-2024 uniquement</span>

In [ ]:
sample_df = []
for i, r in freq_kwh[['p.device_id', 'payload_metric_id']].iterrows():
    device_id = r['p.device_id']
    sensor_id = r['payload_metric_id']
    sample = client.query_df(f"""
        SELECT p.device_id, p.payload_metric_id, payload_timestamp, payload_value, payload_source
        FROM {releves} p
        WHERE p.device_id = '{device_id}'
            AND p.payload_metric_id = '{sensor_id}'
            AND p.payload_timestamp > toDateTime('2024-01-01 01:00:00')
        settings max_rows_to_read = {maxi}""")
    sample_df.append(sample)
sample_df = pd.concat(sample_df)
sample_df["kwh_value"] = sample.payload_value.apply(int)

def tag_continuous_seq(smp, first_id, grp_by):  
    tags_df = []    
    s_cnt = first_id
    for i, s in smp.groupby(grp_by):
        delta_s = pd.DataFrame(s.payload_timestamp.diff().rename('delta'))
        missed = s[delta_s.delta > dt.timedelta(seconds=300)]
        j = s.index[0]
        if missed.empty:
            s[["s_id"]] = s_cnt
            s_cnt += 1
        for i, row in missed.iterrows():
            s.loc[j:i-1, "s_id"] = s_cnt
            s_cnt +=1
            j = i
        if not missed.empty:
            s.loc[j:, "s_id"] = s_cnt
            s_cnt +=1
        tags_df.append(s)
    return pd.concat(tags_df)

sample_df = tag_continuous_seq(sample_df, 0, ['device_id', 'payload_metric_id'])

sequences = sample_df.groupby('s_id').agg(
    couverture=('payload_timestamp', lambda x: x.max() - x.min())
)

In [ ]:
max_val = sequences.couverture.max()
tickvals = [0, 60*60*24,
            86400*30, 2592000*6, 2592000*12, 37576200]
ticktext = ['< 1 journée', '(1j - 1 mois])', '(1 mois - 6 mois])', '(6 mois - 1 an])', 
            '> 1 ans']
sequences["interval"] = pd.cut(sequences["couverture"].apply(lambda x: x.total_seconds()), bins=tickvals, labels=ticktext)
counts = sequences["interval"].value_counts(sort=False)

fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': pink}
    )
)
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Couverture des relevés par séquence continue de données",
                  xaxis_title="Couverture", 
                  yaxis_title="Nombre de séquences continues",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

## <span style="color:green"> 5 - Délai moyen d'ingestion des relevés </span>

In [ ]:
df_lag = client.query_df(f"""
    SELECT d.device_type, p.device_id, p.payload_metric_id, avg(p.ingested_at - p.payload_timestamp) AS avgLag 
    FROM {releves} p
    INNER JOIN {equipements} d 
        ON p.device_id = d.device_id
    WHERE p.payload_source = '{source}'
        AND d.device_type = '{famille}'
    GROUP BY 
        d.device_type, p.device_id, p.payload_metric_id
    ORDER BY avgLag DESC
    settings max_rows_to_read = {maxi}""")     
df_lag

In [ ]:
df_lag['timedelta'] = df_lag.avgLag.apply(lambda x: dt.timedelta(seconds=x))
max_val = df_lag.avgLag.max()
tickvals = [86400, 86400*5, 86400*15, 86400*30, max_val]
ticktext = ['(1jour - 5jours])', '(5jours - 15jours])', 
           '(15jours - 30jours])', '> 1mois']
df_lag["interval"] = pd.cut(df_lag["avgLag"], bins=tickvals, labels=ticktext)
counts = df_lag["interval"].value_counts(sort=False)

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': blue}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Délai moyen d'ingestion des relevés par couple 'equipement-capteur'",
                  xaxis_title="Délai moyen", 
                  yaxis_title="Nombre de couples <br> equipement-capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
df_lag_fam = pd.DataFrame(df_lag.groupby('device_type').avgLag.mean())
df_lag_fam["interval"] = pd.cut(df_lag_fam["avgLag"], bins=tickvals, labels=ticktext)
counts = df_lag_fam["interval"].value_counts(sort=False)
fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': blue}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Délai moyen d'ingestion des relevés par couple 'equipement-capteur'",
                  xaxis_title="Délai moyen", 
                  yaxis_title="Nombre de couples <br> equipement-capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

# <span style="color:purple"> 4 - Données issues du capteur "puissance puissance" </span>

In [ ]:
# capteur dont le nom est '(Cons) Courbe de charge de puissance puissance...'
capt_puissance = '6454b98a0dae31f561c6f68f'

## <span style="color:green">1 - Volume de données </span>

In [ ]:
df_puissance = client.query_df(f"""
                    SELECT p.device_id, count(p.payload_timestamp) nbPay, count(p.payload_value) nbVal
                    FROM {releves} p
                    WHERE p.payload_metric_id = '{capt_puissance}'
                    GROUP BY p.device_id
                    settings max_rows_to_read = {maxi};""")
df_puissance

## <span style="color:green">2 - Répartition des valeurs (clustering) </span>

In [ ]:
df_W = client.query_df(f"""
SELECT
        p.device_id,
        quantileExact(0.25)(toFloat64(p.payload_value)) AS q1,
        quantileExact(0.5)(toFloat64(p.payload_value)) AS median,
        quantileExact(0.75)(toFloat64(p.payload_value)) AS q3,                
        avg(toFloat64(p.payload_value)) AS mean
FROM {releves} AS p
WHERE p.payload_metric_id='{capt_puissance}'
GROUP BY p.device_id
settings max_rows_to_read={maxi};""")

In [ ]:
# merge with entity_name info
pas = 5000
n0 = 0
n1 = n0+pas
list_entity = []
while n1<79138:    
    dev_list = df_W.device_id.to_list()
    dev_list_str = ", ".join([f"'{d}'" for d in dev_list[n0:n1]])
    
    df_entity = client.query_df(f"""SELECT c.entity_name, d.device_id
            FROM {equipements} d
            JOIN {clients} c
                ON d.iot_entity = c.entity_id
            WHERE d.device_id IN ({dev_list_str})
            settings max_rows_to_read={maxi};""")
    list_entity.append(df_entity)
    n0 = n1
    n1 = n0+pas
df_entity = pd.concat(list_entity)
#merge stats device with client
df_W = df_W.merge(df_entity, on='device_id')

In [ ]:
df_W

In [ ]:
# from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
# import plotly.figure_factory as ff
CUT = 5e5
grp_W = df_W.drop(['device_id'], axis=1).groupby('entity_name').mean()
Z = linkage(grp_W.values, method='ward')

# Création du dendrogramme interactif
fig = ff.create_dendrogram(grp_W.values, labels=grp_W.index.tolist(), linkagefun=lambda x: Z)
fig.add_shape(
    type="line",
    x0=0, x1=1, xref='paper',  # x en proportion de la figure (0 à 1)
    y0=CUT, y1=CUT, yref='y',      # y = valeur sur l’axe Y
    line=dict(color="red", width=2, dash="dash")
)
fig.update_layout(title=f"Dendrogramme  des valeurs du capteur puissance puissance par client",
                  xaxis_title="Clients",\
                    yaxis_title="Hauteur/Distance",\
                    font_family="Arial", font_color=black, \
                    title_font_family="Times New Roman", \
                    title_font_color=black, width=1000, \
                    height=400, template="simple_white")
fig.update_xaxes(visible=False)

# Affichage
fig.show()

clusters = fcluster(Z, t=CUT, criterion='distance')
grp_W['clusters'] = clusters

In [ ]:
box_kwh = grp_W.reset_index(drop=True).groupby("clusters").mean()
count_kwh = grp_W.reset_index(drop=True).groupby("clusters").count()
fig = go.Figure()
fig.add_trace(go.Box(name="Box", x=box_kwh.index, q1=box_kwh.q1, median=box_kwh['median'],
                  q3=box_kwh.q3, mean=box_kwh['mean']))
fig.add_trace(go.Bar(name="Nombre de clients", x=count_kwh.index, 
                     y=count_kwh.q1, marker_color="orange", opacity=0.3, yaxis="y2"))
fig.update_yaxes(type="log", range=(3,7))
fig.update_layout(title=f"Répartition des valeurs des relevés par couples équipement-capteur", \
                  xaxis_title="Groupe", yaxis=dict(title="Valeur du relevé (kWh)", type="log", range=[3, 7]),\
                  yaxis2=dict(title="Nombre de clients", overlaying="y", side="right", showgrid=False),\
                  font_family="Arial", font_color=black, title_font_family="Times New Roman",\
                  title_font_color=black, width=1000, height=400, template="simple_white",\
                  legend=dict(x=0.02, y=0.98))    
fig.show()

## <span style="color:green">3 - Fréquence par PDL </span>

In [ ]:
df_freq = client.query_df(f"""
    SELECT p.device_id, arrayReduce('avg', diffs) AS avgDiff
    FROM(
        SELECT p.device_id,
            arrayDifference(arraySort(groupArray(p.payload_timestamp))) AS diffs
        FROM {releves} p
        WHERE p.payload_metric_id='{capt_puissance}'
            AND payload_timestamp > toDateTime('2025-01-01 00:00:00')
        GROUP BY p.device_id) p
    ORDER BY avgDiff DESC
    settings max_rows_to_read = {maxi}""")

In [ ]:
df_freq['timedelta'] = df_freq.avgDiff.apply(lambda x: dt.timedelta(seconds=x))
max_val = df_freq.head(100).avgDiff.max()
tickvals = [-float('inf'), 1, 60, 
            60*5, 60*10, 60*30, 60*45, 
            3600, 3600*6, 3600*12, 
            86400, max_val]
ticktext = ['< 1s', '< 1mn', '(1mn - 5mn]', '(5mn - 10mn]', '(10mn - 30mn]', 
           '(30mn - 45mn]', '(45mn - 1h]', '(1h - 6h]', '(6h - 12h]', 
           '(12h - 1jour]', '> 1jour']
df_freq["interval"] = pd.cut(df_freq["avgDiff"], bins=tickvals, labels=ticktext)
counts = df_freq["interval"].value_counts(sort=False)

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': pink}
    )
)
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Fréquence moyenne des relevés par PDL sur le capteur 'puissance-puissance'",
                  xaxis_title="Fréquence moyenne", 
                  yaxis_title="Nombre de PDL",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

## <span style="color:green">3 - Complétude des relevés PDL par client </span>

### <span style="color:blue">a) Pour les PDL de fréquence comprise entre 10 et 30 mn </span>

In [ ]:
pas = 1000
n0 = 0
n1 = n0+pas
dev_list = df_freq[df_freq.interval == '(10mn - 30mn]'].device_id.to_list()
df_nbPay = []
while n1<3000:    
    dev_list_str = ", ".join([f"'{d}'" for d in dev_list[n0:n1]])
    df = client.query_df(f"""
            SELECT p.device_id, min(p.payload_timestamp) debut, max(p.payload_timestamp) fin, count(p.payload_timestamp) nbPay
            FROM {releves} p
            WHERE p.payload_metric_id='{capt_puissance}'
                AND payload_timestamp > toDateTime('2024-01-01 00:00:00')
                AND device_id IN ({dev_list_str})
            GROUP BY p.device_id
        settings max_rows_to_read = {maxi}""")
    df_nbPay.append(df)
    n0 = n1
    n1 = n0+pas
df_nbPay = pd.concat(df_nbPay)

In [ ]:
df_nbPay['nbTheorique'] = (((df_nbPay.fin - df_nbPay.debut)/pd.Timedelta("15min"))+1).astype(int)
df_nbPay['%missing'] = (df_nbPay['nbTheorique'] - df_nbPay['nbPay'])/df_nbPay['nbPay']*100

In [ ]:
# df_nbPay = df_nbPay.merge(df_freq, on='device_id')
df_nbPay.sort_values('diff_releves').sort_values('%missing')

<span style="color:red">
Lorsque supérieur à 70% de données manquantes on suppose que la fréquence est trop élevée. Il s'agit donc sûrement de fréquence à 30mn
</span>

In [ ]:
df_nbPay[df_nbPay['%missing'] < 90]

### <span style="color:blue">b) Tentative d'arrondissement des fréquence pour obtenir la bonne valeur </span>

In [ ]:
def arrondir_timedelta64_minute(delta):
    # Convertir en secondes (float)
    total_seconds = delta / np.timedelta64(1, 's')
    # Arrondir à la minute la plus proche
    minutes_arrondies = np.round(total_seconds / 60)
    # Revenir à un timedelta64 en minutes
    return minutes_arrondies.astype('timedelta64[m]')

In [ ]:
df_freq["rnd_min_delta"] = df_freq['timedelta'].apply(lambda x: arrondir_timedelta64_minute(x))

In [ ]:
df_freq[df_freq.interval=='(10mn - 30mn]']

In [ ]:
minute_df = df_freq[df_freq.interval.isin(['(1mn - 5mn]', '(5mn - 10mn]'])]
minute_df["rnd_min_delta"] = minute_df['timedelta'].apply(lambda x: arrondir_timedelta64_minute(x))

In [ ]:
minute_df[minute_df.interval=="(5mn - 10mn]"].sort_values('timedelta')[50:100]

Imputation de donnees manquantes

In [ ]:
df_freq

In [ ]:
def arrondir_timedelta64_minute(delta):
    # Convertir en secondes (float)
    total_seconds = delta / np.timedelta64(1, 's')
    # Arrondir à la minute la plus proche
    minutes_arrondies = np.round(total_seconds / 60)
    # Revenir à un timedelta64 en minutes
    return minutes_arrondies.astype('timedelta64[m]')

In [ ]:
df_freq["rnd_min_delta"] = df_freq['timedelta'].apply(lambda x: arrondir_timedelta64_minute(x))

In [ ]:
df_freq[df_freq.interval=='(10mn - 30mn]']

In [ ]:
minute_df = df_freq[df_freq.interval.isin(['(1mn - 5mn]', '(5mn - 10mn]'])]
minute_df["rnd_min_delta"] = minute_df['timedelta'].apply(lambda x: arrondir_timedelta64_minute(x))

In [ ]:
df_freq = client.query_df(f"""
    SELECT p.device_id, arrayReduce('avg', diffs) AS avgDiff
    FROM(
        SELECT p.device_id,
            arrayDifference(arraySort(groupArray(p.payload_timestamp))) AS diffs
        FROM {releves} p
        WHERE p.payload_metric_id='{capt_puissance}'
            AND payload_timestamp > toDateTime('2025-01-01 00:00:00')
            AND device_id IN ({ma_liste})
        GROUP BY p.device_id) p
    ORDER BY avgDiff DESC
    settings max_rows_to_read = {maxi}""")

In [ ]:
# Parmi les PDL 5- 10mn chercher ceux dont la différence de timestamp est supérieure à 10mn
sample = client.query_df(f"""
            SELECT p.device_id, p.payload_timestamp, p.payload_value
            FROM {releves} p
            WHERE p.payload_metric_id='{capt_puissance}'
                AND payload_timestamp > toDateTime('2024-01-01 00:00:00')
                AND device_id = '658d9654d3f061b7ac432597'
        settings max_rows_to_read = {maxi}""")

In [ ]:
sample

In [ ]:
print('30 minutes:')
print(sample[sample.freq == np.timedelta64(60*30,'s')].payload_timestamp.min())
print(sample[sample.freq == np.timedelta64(60*30,'s')].payload_timestamp.max())
print('15 minutes:')
print(sample[sample.freq == np.timedelta64(60*15,'s')].payload_timestamp.min())
print(sample[sample.freq == np.timedelta64(60*15,'s')].payload_timestamp.max())

# <span style="color:purple"> 5 - Tableaux synthétiques </span>

## <span style="color:green"> 1 - Par famille d'équipements </span>

In [ ]:
df_familles = client.query_df(f"""
SELECT d.device_type, count(distinct p.entity_id) ent, count(p.payload_timestamp) nbPay
FROM {releves} p
INNER JOIN (SELECT device_type, device_id
		FROM {equipements}) d	
ON p.device_id = d.device_id
WHERE p.payload_source = '{source}'
GROUP BY d.device_type
settings max_rows_to_read = {maxi};""")

In [ ]:
def sql_quote(s):
    return s.replace("'", "\\'")
 
synth_df = []
for i in df_familles.device_type.drop_duplicates().to_list():
    famille = sql_quote(i)
    df = client.query_df(f"""
        SELECT p.device_id, p.payload_metric_id, arrayReduce('avg', diffs) AS avgDiff 
        FROM(
            SELECT 
                p.device_id, 
                p.payload_metric_id, 
                arrayDifference(arraySort(groupArray(p.payload_timestamp))) AS diffs
            FROM {releves} p
            INNER JOIN {equipements} d 
            ON p.device_id = d.device_id
            WHERE d.device_type = '{famille}' AND p.payload_source = '{source}'
        GROUP BY 
            p.device_id, 
            p.payload_metric_id) p
        ORDER BY avgDiff DESC
        settings max_rows_to_read = {maxi}""")
    avgFreq = str(dt.timedelta(seconds=df.avgDiff.mean()))
    minFreq = str(dt.timedelta(seconds=df.avgDiff.min()))
    maxFreq = str(dt.timedelta(seconds=df.avgDiff.max()))
    nbcoup = df.shape[0]
    nbSens = df.payload_metric_id.drop_duplicates().shape[0]
    nbdev = df.device_id.drop_duplicates().shape[0]
    synth_df.append({'Famille':famille, "Nombre de capteurs":nbSens, "Nombre d'équipements":nbdev, 
                     'Couples_equ-cap':nbcoup, 'Frequence_moyenne':avgFreq,
                     'Frequence_min':minFreq, 'Frequence_max':maxFreq})

In [ ]:
df1 = pd.read_csv('./data/enedis_synthese_par_famille.csv')
df1.drop_duplicates(inplace=True)
df1 = df1.merge(df_familles.rename({'ent':'Nombre de clients', 'nbPay':'Nombre de relevés'}, axis=1), right_on='device_type', left_on='Famille')

In [ ]:
df1.sort_values('Nombre de capteurs', ascending=False)

# <span style="color:purple"> 6 - Annexes </span>

## <span style="color:green"> 1 — Recherche des **équipements inactifs** (pas de relevés dans la collection payload) </span>
Parmi les équipements appartenant à l'une des familles qui se manifestent dans les payloads enedis

In [ ]:
# Liste des familles qui se manifestent dans les payloads enedis
df_familles = client.query_df(f"""
SELECT DISTINCT d.device_type
FROM {releves} p
WHERE p.payload_source = '{source}'
INNER JOIN (SELECT device_type, device_id
		FROM {equipements}) d	
ON p.device_id = d.device_id
settings max_rows_to_read = {maxi};""")

In [ ]:
list_fam = df_familles.device_type.to_list()
def sql_quote(s):
    return "'" + s.replace("'", "\\'") + "'"
list_fam = ', '.join(sql_quote(x) for x in list_fam)

In [ ]:
cnt_dev = client.query_df(f"""
    SELECT DISTINCT d.device_id, d.device_type
    FROM {equipements} d
    WHERE d.device_type IN ({list_fam})
    SETTINGS max_rows_to_read = {maxi}""")

df_null = client.query_df(f"""
    SELECT d.device_id, d.device_type, f.device_id pay
    FROM {equipements} d
    LEFT JOIN(
    	SELECT DISTINCT device_id
    	FROM {releves}) f
    ON d.device_id = f.device_id
    WHERE d.device_type IN ({list_fam}) AND pay IS NULL
    SETTINGS join_use_nulls = 1, max_rows_to_read = {maxi}""")

print(f"{df_null.shape[0]} équipements parmi les familles suivantes n'ont pas de relevé dans la table payload:\n\n{list_fam}\n")

print(f"Cela représente {int(df_null.shape[0]/cnt_dev.shape[0]*100)} % des équipements appartenant à cette liste de familles")

##### **Nombre d'équipements par famille n'ayant aucun relevé dans la table payload**

In [ ]:
pd.DataFrame(df_null.groupby('device_type').device_id.count().sort_values(ascending=False))

## <span style="color:green"> 2 - Recherche des **capteurs dupliqués** (doublons créés par identifiants modifié) </span>

In [ ]:
list_sens = client.query_df(f"""
SELECT DISTINCT payload_metric_id
FROM {releves}
WHERE payload_source = '{source}' AND match(payload_metric_id, '^[a-z0-9]+$')
SETTINGS max_rows_to_read = {maxi};""")

In [ ]:
# list_sens['length'] = list_sens.payload_metric_id.apply(len)
list_sens[list_sens.length >24].payload_metric_id.apply(lambda x: x[:24])

In [ ]:
doubles = list_sens[list_sens.length >24].payload_metric_id.apply(lambda x: x[:24]).unique()

In [ ]:
for i in doubles:
    print(i+'[0-9]+')

In [ ]:
df_double = []
for i in doubles:
    df_double.append(
        client.query_df(f"""
            SELECT device_id, payload_timestamp, count(payload_value) nbPay
            FROM {releves}
            WHERE payload_source = '{source}' 
            	AND match(payload_metric_id, '{i+'[0-9]+'}')
            GROUP BY device_id, payload_timestamp
            ORDER BY nbPay DESC, payload_timestamp 
            SETTINGS max_rows_to_read = 10000000000000;"""))
df_double = pd.concat(df_double)